Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 03b · Foundation validation

**Question:** Are the inputs consistent enough for risk measurement?

The shared loader, event ledger, same-security rules, manifests and frozen calendar are checked before model interpretation. This stage validates and corrects inputs; it is not a statistical normalization step.

Research report · Method and code guide · Evidence map


**Chapter route**

Measurement and interpretation principles · Data and design context · Read and run this notebook · Calculation source: foundation.corrections · Calculation source: foundation.eligibility · Calculation source: foundation.inventory · Calculation source: foundation.validation · Calculation source: event sensitivity · Results, examples and interpretation · 1. Reproduction with corrections off, and what the corrections change · 2. Correction sensitivity · 3. Panels, cutoffs and the formation calendar · 4. Securities eligible at each formation · 5. The extreme-event ledger · 6. Nonpositive market cap · 7. The 2026-10-13 leftovers · 8. Ignore rules and manifest completeness · Conclusions · Visual evidence · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

Validation separates exact reproduction with corrections off from the impact of accepted corrections. Fingerprints establish lineage; same-security rules establish which history belongs to the held security. A correction sensitivity changes a documented input decision while holding the risk measurement method fixed.


## Data and design context

This notebook checks the shared data foundation (docs/STUDY_SPEC.md section 3) before any estimation uses it. Every check is defined in the calculation cells below; the notebook calls them, shows the result and saves each table under `results/foundation/` with a manifest (the one table built from FMP market cap goes under `results/licensed/foundation/`, where the CSV is git-ignored). It reads the frozen 2026-10-12 vintage through the loader and never the 2026-10-13 leftovers, which are only listed.

The order follows the acceptance rule of the specification: first that the loader with corrections switched off reproduces the processed tables that existed before it, then exactly what the corrections change, then the panels, cutoffs, calendar and eligibility counts that every later task relies on, and last the housekeeping checks on market cap, leftovers, manifests and ignore rules.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version.


## Calculation source: foundation.corrections

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Extreme-event corrections: an evidenced ledger applied to the cleaned adjusted-close panel, never to raw files.

The ledger lives at data/corrections/extreme_event_ledger_2026-09-30.csv. If the file does not exist, no correction is
applied. Each row is one decision about one ticker; a ticker may have several rows.

Ledger columns (CSV, one header row; dates as YYYY-MM-DD; blank where a column does not apply)
- ticker: the universe ticker the row applies to.
- event_date: the session of the daily return under review (the return from the previous session's price to this one).
- classification: one of verified_market_move, verified_vendor_error, corporate_action_discrepancy, unresolved.
- treatment: one of retain, back_adjust_before_date, replace_price, quarantine_price, quarantine_return.
- factor: back_adjust_before_date only. Every adjusted close strictly before event_date is multiplied by it. If blank,
  it is solved from corrected_return so that the event_date return equals corrected_return.
- price_date: replace_price only. The session whose adjusted close is replaced.
- price_value: replace_price only. The replacement adjusted close, quoted on the basis of the downloaded panel.
- quarantine_dates: quarantine_price and quarantine_return. Semicolon-separated sessions; blank means event_date alone.
  For quarantine_price these are price dates (the price is set missing, so the returns on that session and the next
  are both missing); for quarantine_return they are return dates (that return is missing; prices are untouched).
- corrected_return: the evidenced daily simple return on event_date. Required when back_adjust_before_date has no
  factor; whenever it is given for back_adjust_before_date or replace_price, the corrected panel must reproduce it.
- evidence_reference: where the evidence is recorded (a document section, filing or note). Required for every row.

Which treatments each classification allows (docs/STUDY_SPEC.md section 3)
- verified_market_move: retain (no change).
- verified_vendor_error: replace_price or back_adjust_before_date (an evidenced correction).
- corporate_action_discrepancy: back_adjust_before_date or replace_price (reconstruct the distribution-inclusive return).
- unresolved: quarantine_price or quarantine_return (disclosed, never filled).

Order of application, so the result does not depend on row order: replacements first (their values are on the
downloaded basis), then back adjustments in date order, then quarantines. Missing sessions are never bridged: a
return always needs the price of the session immediately before it.
"""

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from quiet_index_500 import settings

ledger_file_name = f"extreme_event_ledger_{settings.holdings_date}.csv"
ledger_columns = ("ticker", "event_date", "classification", "treatment", "factor", "price_date", "price_value",
                  "quarantine_dates", "corrected_return", "evidence_reference")
treatments_allowed_by_classification = {
    "verified_market_move": {"retain"},
    "verified_vendor_error": {"replace_price", "back_adjust_before_date"},
    "corporate_action_discrepancy": {"back_adjust_before_date", "replace_price"},
    "unresolved": {"quarantine_price", "quarantine_return"},
}
corrected_return_tolerance = 1e-10  # how closely a corrected panel must reproduce an evidenced return


**extreme_event_ledger_path**

data/corrections/extreme_event_ledger_2026-09-30.csv (it may not exist).


In [2]:
def extreme_event_ledger_path():
    """data/corrections/extreme_event_ledger_2026-09-30.csv (it may not exist)."""
    return settings.corrections_folder / ledger_file_name


**empty_extreme_event_ledger**

A ledger with the documented columns and no rows: what an absent ledger means.


In [3]:
def empty_extreme_event_ledger():
    """A ledger with the documented columns and no rows: what an absent ledger means."""
    return pd.DataFrame({column_name: pd.Series(dtype="object") for column_name in ledger_columns})


**read_extreme_event_ledger**

Read and validate the ledger; an absent file returns an empty ledger (no corrections).


In [4]:
def read_extreme_event_ledger(ledger_path=None):
    """Read and validate the ledger; an absent file returns an empty ledger (no corrections)."""
    ledger_path = extreme_event_ledger_path() if ledger_path is None else ledger_path
    if not ledger_path.exists():
        return empty_extreme_event_ledger()
    ledger = pd.read_csv(ledger_path, dtype={"ticker": str, "classification": str, "treatment": str,
                                             "quarantine_dates": str, "evidence_reference": str})
    validate_extreme_event_ledger(ledger)
    return ledger


**validate_extreme_event_ledger**

Raise ValueError if a column is missing, a value is outside the documented sets, or a required field is blank.


In [5]:
def validate_extreme_event_ledger(ledger):
    """Raise ValueError if a column is missing, a value is outside the documented sets, or a required field is blank."""
    missing_columns = [column_name for column_name in ledger_columns if column_name not in ledger.columns]
    if missing_columns:
        raise ValueError(f"ledger is missing columns {missing_columns}")
    for row_number, ledger_row in ledger.iterrows():
        row_label = f"ledger row {row_number} ({ledger_row['ticker']}, {ledger_row['event_date']})"
        if ledger_row["classification"] not in treatments_allowed_by_classification:
            raise ValueError(f"{row_label}: unknown classification {ledger_row['classification']!r}")
        if ledger_row["treatment"] not in treatments_allowed_by_classification[ledger_row["classification"]]:
            raise ValueError(f"{row_label}: treatment {ledger_row['treatment']!r} is not allowed for {ledger_row['classification']}")
        if pd.isna(ledger_row["evidence_reference"]) or not str(ledger_row["evidence_reference"]).strip():
            raise ValueError(f"{row_label}: evidence_reference is required")
        if pd.isna(ledger_row["event_date"]):
            raise ValueError(f"{row_label}: event_date is required")
        if ledger_row["treatment"] == "back_adjust_before_date" and pd.isna(ledger_row["factor"]) and pd.isna(ledger_row["corrected_return"]):
            raise ValueError(f"{row_label}: back_adjust_before_date needs a factor or a corrected_return")
        if ledger_row["treatment"] == "back_adjust_before_date" and pd.notna(ledger_row["factor"]) and not float(ledger_row["factor"]) > 0:
            raise ValueError(f"{row_label}: factor must be positive")
        if ledger_row["treatment"] == "replace_price" and (pd.isna(ledger_row["price_date"]) or pd.isna(ledger_row["price_value"])):
            raise ValueError(f"{row_label}: replace_price needs price_date and price_value")
        if ledger_row["treatment"] == "replace_price" and not float(ledger_row["price_value"]) > 0:
            raise ValueError(f"{row_label}: price_value must be positive")


**_quarantine_sessions**

The sessions a quarantine row names: quarantine_dates split on ';', or event_date when blank.


In [6]:
def _quarantine_sessions(ledger_row):
    """The sessions a quarantine row names: quarantine_dates split on ';', or event_date when blank."""
    if pd.isna(ledger_row["quarantine_dates"]) or not str(ledger_row["quarantine_dates"]).strip():
        return [pd.Timestamp(ledger_row["event_date"])]
    return [pd.Timestamp(date_text.strip()) for date_text in str(ledger_row["quarantine_dates"]).split(";") if date_text.strip()]


**_require_session**

Raise unless the date is a session of the panel.


In [7]:
def _require_session(sessions, session_date, row_label):
    """Raise unless the date is a session of the panel."""
    if session_date not in sessions:
        raise ValueError(f"{row_label}: {session_date.date()} is not a session of the price panel")


**CorrectedAdjustedClose**

What applying the ledger produces: the corrected panel, the quarantine masks and a log of what was done.


In [8]:
@dataclass
class CorrectedAdjustedClose:
    """What applying the ledger produces: the corrected panel, the quarantine masks and a log of what was done."""
    adjusted_close: pd.DataFrame
    quarantined_prices: pd.DataFrame
    quarantined_returns: pd.DataFrame
    applied_corrections: pd.DataFrame = field(default_factory=pd.DataFrame)


**apply_extreme_event_ledger**

Apply the ledger to a cleaned adjusted-close panel (date index, one column per ticker) and return the result.

The input is not modified. quarantined_prices marks prices set missing; quarantined_returns marks every return made
missing by a quarantine: the named return dates, and for each quarantined price its own session and the next one.


In [9]:
def apply_extreme_event_ledger(adjusted_close, ledger):
    """Apply the ledger to a cleaned adjusted-close panel (date index, one column per ticker) and return the result.

    The input is not modified. quarantined_prices marks prices set missing; quarantined_returns marks every return made
    missing by a quarantine: the named return dates, and for each quarantined price its own session and the next one.
    """
    corrected_adjusted_close = adjusted_close.copy()
    sessions = corrected_adjusted_close.index
    quarantined_prices = pd.DataFrame(False, index=sessions, columns=corrected_adjusted_close.columns)
    quarantined_returns = quarantined_prices.copy()
    applied_log_rows = []
    if len(ledger) == 0:
        return CorrectedAdjustedClose(corrected_adjusted_close, quarantined_prices, quarantined_returns, pd.DataFrame(applied_log_rows))

    validate_extreme_event_ledger(ledger)
    treatment_order = {"replace_price": 0, "back_adjust_before_date": 1, "quarantine_price": 2, "quarantine_return": 3, "retain": 4}
    ordered_ledger = ledger.assign(
        treatment_rank=ledger["treatment"].map(treatment_order),
        event_timestamp=pd.to_datetime(ledger["event_date"]),
    ).sort_values(["treatment_rank", "event_timestamp"], kind="stable")

    for row_number, ledger_row in ordered_ledger.iterrows():
        ticker = ledger_row["ticker"]
        event_date = ledger_row["event_timestamp"]
        row_label = f"ledger row {row_number} ({ticker}, {event_date.date()})"
        if ticker not in corrected_adjusted_close.columns:
            raise ValueError(f"{row_label}: ticker is not in the price panel")
        if event_date > sessions.max():  # an event after the loaded window has nothing to act on here
            continue
        _require_session(sessions, event_date, row_label)
        treatment = ledger_row["treatment"]

        if treatment == "replace_price":
            price_date = pd.Timestamp(ledger_row["price_date"])
            _require_session(sessions, price_date, row_label)
            price_before_replacement = corrected_adjusted_close.at[price_date, ticker]
            corrected_adjusted_close.at[price_date, ticker] = float(ledger_row["price_value"])  # on the downloaded basis, before any back adjustment
            applied_log_rows.append({"ticker": ticker, "event_date": event_date, "treatment": treatment, "detail": f"{price_date.date()}: {price_before_replacement} -> {float(ledger_row['price_value'])}"})

        elif treatment == "back_adjust_before_date":
            event_position = sessions.get_loc(event_date)
            if event_position == 0:
                raise ValueError(f"{row_label}: no session before event_date to adjust")
            if pd.notna(ledger_row["factor"]):
                back_adjustment_factor = float(ledger_row["factor"])
            else:  # solve the factor so that P_t / (factor * P_t-1) - 1 equals the evidenced return
                price_on_event_date = corrected_adjusted_close.iat[event_position, corrected_adjusted_close.columns.get_loc(ticker)]
                price_on_previous_session = corrected_adjusted_close.iat[event_position - 1, corrected_adjusted_close.columns.get_loc(ticker)]
                if pd.isna(price_on_event_date) or pd.isna(price_on_previous_session):
                    raise ValueError(f"{row_label}: a factor cannot be solved without prices on the event session and the session before it")
                back_adjustment_factor = price_on_event_date / ((1 + float(ledger_row["corrected_return"])) * price_on_previous_session)
            sessions_before_event = sessions < event_date
            corrected_adjusted_close.loc[sessions_before_event, ticker] = corrected_adjusted_close.loc[sessions_before_event, ticker] * back_adjustment_factor
            applied_log_rows.append({"ticker": ticker, "event_date": event_date, "treatment": treatment, "detail": f"prices before {event_date.date()} x {back_adjustment_factor:.10g}"})

        elif treatment == "quarantine_price":
            for price_date in _quarantine_sessions(ledger_row):
                _require_session(sessions, price_date, row_label)
                price_position = sessions.get_loc(price_date)
                corrected_adjusted_close.at[price_date, ticker] = np.nan
                quarantined_prices.at[price_date, ticker] = True
                quarantined_returns.iat[price_position, quarantined_returns.columns.get_loc(ticker)] = True  # the return into the suspect price
                if price_position + 1 < len(sessions):
                    quarantined_returns.iat[price_position + 1, quarantined_returns.columns.get_loc(ticker)] = True  # and the return out of it
                applied_log_rows.append({"ticker": ticker, "event_date": event_date, "treatment": treatment, "detail": f"price {price_date.date()} set missing"})

        elif treatment == "quarantine_return":
            for return_date in _quarantine_sessions(ledger_row):
                _require_session(sessions, return_date, row_label)
                quarantined_returns.at[return_date, ticker] = True
                applied_log_rows.append({"ticker": ticker, "event_date": event_date, "treatment": treatment, "detail": f"return {return_date.date()} set missing"})

        else:  # retain: a verified market move stays as downloaded
            applied_log_rows.append({"ticker": ticker, "event_date": event_date, "treatment": treatment, "detail": "unchanged"})

    # every evidenced return given for a price correction must come out of the corrected panel exactly
    for row_number, ledger_row in ordered_ledger.iterrows():
        if ledger_row["treatment"] not in ("back_adjust_before_date", "replace_price") or pd.isna(ledger_row["corrected_return"]):
            continue
        event_date = ledger_row["event_timestamp"]
        if event_date > sessions.max():
            continue
        event_position = sessions.get_loc(event_date)
        ticker_prices = corrected_adjusted_close[ledger_row["ticker"]]
        reproduced_return = ticker_prices.iloc[event_position] / ticker_prices.iloc[event_position - 1] - 1  # the session immediately before, never a bridge
        if not abs(reproduced_return - float(ledger_row["corrected_return"])) <= corrected_return_tolerance:
            raise ValueError(f"ledger row {row_number} ({ledger_row['ticker']}, {event_date.date()}): corrected panel gives {reproduced_return}, "
                             f"ledger says {ledger_row['corrected_return']}")

    return CorrectedAdjustedClose(corrected_adjusted_close, quarantined_prices, quarantined_returns, pd.DataFrame(applied_log_rows))


## Calculation source: foundation.eligibility

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [10]:
"""Eligibility of an estimation window, with the reason when it fails, so every table can carry exclusions.

A window is eligible when it has at least the minimum number of complete joint observations (sessions on which every
required series has a value) and it contains no session before the security's same-security start
(docs/STUDY_SPEC.md section 4). A quarantined return is one missing observation: it never enters the count, but it
does not by itself disqualify the window. Each result reports how many quarantined returns the window holds, so the
exclusion stays visible (Sai, 14 Oct 2026, following the master plan's "quarantine the affected return").

Reasons, checked in this order (the first that applies is reported)
- before_same_security_start: the window reaches back before the security's same-security history.
- no_prices: the security's own series has no value anywhere in the window.
- insufficient_observations: fewer complete joint observations than the minimum (quarantined days count as missing).
- eligible: none of the above.
"""

import numpy as np
import pandas as pd

reason_eligible = "eligible"
eligibility_reasons = ("before_same_security_start", "no_prices", "insufficient_observations", reason_eligible)


**complete_joint_observations**

Number of sessions in the window on which every column of required_series has a value.


In [11]:
def complete_joint_observations(required_series, window):
    """Number of sessions in the window on which every column of required_series has a value."""
    required_in_window = required_series.reindex(pd.DatetimeIndex(window))
    if isinstance(required_in_window, pd.Series):
        return int(required_in_window.notna().sum())
    return int(required_in_window.notna().all(axis=1).sum())


**window_eligibility**

Eligibility of one security's window: a dict with observations, quarantined_in_window, eligible and reason.

security_series is the security's own series (its returns, quarantined days already missing); other_required_series
holds the other series the estimate needs on the same days (factors, SPY); the two boolean masks are the security's
columns from PricePanels.


In [12]:
def window_eligibility(security_series, window, minimum_observations, other_required_series=None,
                       before_same_security_start=None, quarantined_returns=None):
    """Eligibility of one security's window: a dict with observations, quarantined_in_window, eligible and reason.

    security_series is the security's own series (its returns, quarantined days already missing); other_required_series
    holds the other series the estimate needs on the same days (factors, SPY); the two boolean masks are the security's
    columns from PricePanels.
    """
    window = pd.DatetimeIndex(window)
    required_series = security_series.to_frame("security")
    if other_required_series is not None:
        required_series = required_series.join(other_required_series.reindex(required_series.index), how="left")
    quarantined_in_window = (int(quarantined_returns.reindex(window, fill_value=False).sum())
                             if quarantined_returns is not None else 0)
    if quarantined_returns is not None:  # a quarantined return never counts, even if a caller passes an unmasked series
        required_series = required_series.mask(quarantined_returns.reindex(required_series.index, fill_value=False), axis=0)
    observations = complete_joint_observations(required_series, window)
    reaches_before_start = before_same_security_start is not None and bool(before_same_security_start.reindex(window, fill_value=False).any())
    has_no_values = not required_series["security"].reindex(window).notna().any()
    if reaches_before_start:
        reason = "before_same_security_start"
    elif has_no_values:
        reason = "no_prices"
    elif observations < minimum_observations:
        reason = "insufficient_observations"
    else:
        reason = reason_eligible
    return {"observations": observations, "quarantined_in_window": quarantined_in_window,
            "eligible": reason == reason_eligible, "reason": reason}


**panel_window_eligibility**

window_eligibility for every column of a panel at once: a ticker-indexed frame with observations,
quarantined_in_window, eligible, reason.


In [13]:
def panel_window_eligibility(security_panel, window, minimum_observations, other_required_series=None,
                             before_same_security_start=None, quarantined_returns=None):
    """window_eligibility for every column of a panel at once: a ticker-indexed frame with observations,
    quarantined_in_window, eligible, reason."""
    window = pd.DatetimeIndex(window)
    panel_in_window = security_panel.reindex(window)
    quarantined_in_window_grid = (quarantined_returns.reindex(index=window, columns=security_panel.columns, fill_value=False).to_numpy()
                                  if quarantined_returns is not None else np.zeros(panel_in_window.shape, dtype=bool))
    security_has_value = panel_in_window.notna().to_numpy() & ~quarantined_in_window_grid  # a quarantined return is missing
    other_series_complete = np.ones(len(window), dtype=bool)
    if other_required_series is not None:
        other_in_window = other_required_series.reindex(window)
        other_series_complete = (other_in_window.notna().all(axis=1) if isinstance(other_in_window, pd.DataFrame) else other_in_window.notna()).to_numpy()
    observations = (security_has_value & other_series_complete[:, None]).sum(axis=0)
    reaches_before_start = (before_same_security_start.reindex(index=window, columns=security_panel.columns, fill_value=False).to_numpy().any(axis=0)
                            if before_same_security_start is not None else np.zeros(security_panel.shape[1], dtype=bool))
    has_no_values = ~security_has_value.any(axis=0)
    reason = np.select(
        [reaches_before_start, has_no_values, observations < minimum_observations],
        ["before_same_security_start", "no_prices", "insufficient_observations"],
        default=reason_eligible,
    )
    return pd.DataFrame({"observations": observations, "quarantined_in_window": quarantined_in_window_grid.sum(axis=0),
                         "eligible": reason == reason_eligible, "reason": reason},
                        index=security_panel.columns).rename_axis("ticker")


## Calculation source: foundation.inventory

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [14]:
"""Inventory of the files left outside the study: the 2026-10-13 price pull in data/raw.

They are listed for the validation report (path, size, sha256) and left in place. Nothing else in the package opens
them; hashing reads their bytes but never parses them.
"""

import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation.manifests import repository_relative_path, sha256_of_file

leftover_vintage_marker = "2026-10-13"  # the pull after the frozen 2026-10-12 vintage


**inventory_leftover_files**

One row per file in folder (default data/raw) whose name contains the marker: path, bytes, sha256.


In [15]:
def inventory_leftover_files(folder=None, marker=leftover_vintage_marker):
    """One row per file in folder (default data/raw) whose name contains the marker: path, bytes, sha256."""
    folder = settings.raw_data_folder if folder is None else folder
    leftover_paths = sorted(file_path for file_path in folder.iterdir() if file_path.is_file() and marker in file_path.name)
    return pd.DataFrame(
        [{"path": repository_relative_path(file_path), "bytes": file_path.stat().st_size, "sha256": sha256_of_file(file_path)}
         for file_path in leftover_paths],
        columns=["path", "bytes", "sha256"],
    )


## Calculation source: foundation.validation

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [16]:
"""Foundation validation: the checks behind notebooks/03b_foundation_validation.ipynb, authored alongside the notebook explanations.

Each function returns a table; the notebook displays it and saves it under results/foundation/ (or
results/licensed/foundation/ when the values come from FMP market cap) with a manifest.

What is checked (docs/STUDY_SPEC.md section 3)
- Reproduction: with corrections off, the build reproduces the processed tables that existed before the shared loader
  (by sha256), except the two flag columns the foundation appended to ticker_data_flags.
- Correction effects: every value that changes between the corrections-off and corrections-on versions of the
  processed tables and of the loader's daily return panel.
- Correction sensitivity: corrections on versus off for the snapshot portfolio return series and, for every
  back-adjusted security, the 504-session volatility and 12-1 momentum at each formation whose window includes the event.
- Panel shapes, cutoffs, eligibility counts per formation, the event ledger against the loader, nonpositive market cap
  handling, manifest completeness and the git ignore rules for licensed and raw files.
"""

import hashlib
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation import build, calendar, corrections, eligibility, loader, manifests

validation_notebook = "notebooks/03b_foundation_validation.ipynb"
results_area = "foundation"

# the tables notebook 02 writes, with what their rows and columns are, so a changed cell can be named by ticker and date
processed_table_layouts = {
    f"same_security_rules_{settings.holdings_date}": {"row": "ticker", "column": "field"},
    f"snapshot_portfolio_daily_returns_through_{settings.price_vintage_last_session}": {"row": "date", "column": "field"},
    f"spy_daily_returns_through_{settings.price_vintage_last_session}": {"row": "date", "column": "field"},
    f"smh_daily_returns_through_{settings.price_vintage_last_session}": {"row": "date", "column": "field"},
    f"portfolio_exclusions_through_{settings.price_vintage_last_session}": {"row": "date_and_ticker", "column": "field"},
    f"ticker_data_flags_through_{settings.price_vintage_last_session}": {"row": "ticker", "column": "field"},
    f"market_cap_same_security_through_{settings.price_vintage_last_session}": {"row": "date", "column": "ticker"},
    f"issuer_market_cap_through_{settings.price_vintage_last_session}": {"row": "date", "column": "cik"},
}
licensed_table_stems = {stem for stem in processed_table_layouts if "market_cap" in stem}  # FMP-derived: values never leave results/licensed
appended_flag_columns = ("market_cap_nonpositive_days", "quarantined_return_days")  # added by the foundation after the original columns
change_columns = ["table", "ticker", "date", "column", "before", "after"]
rounding_noise_tolerance = 1e-12  # a change this small in a return or weight is the last binary digit, not a correction


# ---------- saving ----------


**save_results_table**

Write a validation table to results/foundation/ (or results/licensed/foundation/) with its manifest; returns the path.


In [17]:
def save_results_table(table, file_name, note, input_paths=(), derived_from_licensed_data=False, index=False, extra_fields=None):
    """Write a validation table to results/foundation/ (or results/licensed/foundation/) with its manifest; returns the path."""
    table_path = manifests.results_table_path(results_area, file_name, derived_from_licensed_data)
    table.to_csv(table_path, index=index)
    from quiet_index_500.research_io import study_input_paths
    dependencies = list(dict.fromkeys([*input_paths, *study_input_paths(include_market_cap=True)]))
    source_paths = sorted((settings.repository_root_folder / "src/quiet_index_500/foundation").glob("*.py"))
    manifests.write_manifest(table_path, dependencies, validation_notebook, note=note,
                             source_modules=[settings, *source_paths],
                             extra_fields=extra_fields)
    return table_path


# ---------- reproduction and correction effects ----------


**strip_trailing_csv_fields**

The CSV bytes with the last field_count comma-separated fields removed from every line (no quoted commas assumed).


In [18]:
def strip_trailing_csv_fields(csv_bytes, field_count):
    """The CSV bytes with the last field_count comma-separated fields removed from every line (no quoted commas assumed)."""
    stripped_lines = [line.rsplit(b",", field_count)[0] for line in csv_bytes.split(b"\n")]
    return b"\n".join(stripped_lines)


**read_pre_refactor_hashes**

The sha256 list saved before the loader rewrote data/processed ('<sha256>  <path>' per line) as a frame.


In [19]:
def read_pre_refactor_hashes(hashes_path):
    """The sha256 list saved before the loader rewrote data/processed ('<sha256>  <path>' per line) as a frame."""
    rows = []
    for line in Path(hashes_path).read_text().splitlines():
        if line.strip():
            sha256_text, path_text = line.split(maxsplit=1)
            rows.append({"file": path_text.strip(), "sha256_pre_refactor": sha256_text})
    return pd.DataFrame(rows, columns=["file", "sha256_pre_refactor"])


**build_corrections_off_tables**

Rebuild every table notebook 02 writes with corrections off into output_folder (a scratch folder); returns the paths.


In [20]:
def build_corrections_off_tables(output_folder):
    """Rebuild every table notebook 02 writes with corrections off into output_folder (a scratch folder); returns the paths."""
    return build.build_all_processed_tables(through=settings.price_vintage_last_session, apply_corrections=False,
                                            output_folder=output_folder, built_by=f"{validation_notebook} (corrections-off rebuild)")


**reproduction_check**

Compare each pre-refactor CSV hash with its reproduction: the corrections-off rebuild for tables notebook 02 writes,
the file on disk for the tables it does not write. ticker_data_flags is also compared with its two appended columns
stripped, which must reproduce the old bytes exactly.


In [21]:
def reproduction_check(pre_refactor_hashes, rebuilt_folder, processed_folder=None):
    """Compare each pre-refactor CSV hash with its reproduction: the corrections-off rebuild for tables notebook 02 writes,
    the file on disk for the tables it does not write. ticker_data_flags is also compared with its two appended columns
    stripped, which must reproduce the old bytes exactly.
    """
    processed_folder = settings.processed_data_folder if processed_folder is None else Path(processed_folder)
    rows = []
    for record in pre_refactor_hashes.itertuples():
        file_name = Path(record.file).name
        if not file_name.endswith(".csv"):
            continue  # manifests carry build times and new fields by design; the tables are what must reproduce
        stem = file_name[:-len(".csv")]
        written_by_notebook_02 = stem in processed_table_layouts
        reproduced_path = (Path(rebuilt_folder) if written_by_notebook_02 else processed_folder) / file_name
        reproduced_bytes = reproduced_path.read_bytes()
        reproduced_sha256 = manifests.sha256_of_file(reproduced_path)
        row = {"file": record.file, "sha256_pre_refactor": record.sha256_pre_refactor, "sha256_reproduced": reproduced_sha256,
               "reproduced_from": "corrections-off rebuild" if written_by_notebook_02 else "unchanged file on disk",
               "identical": reproduced_sha256 == record.sha256_pre_refactor,
               "sha256_without_appended_columns": "", "identical_without_appended_columns": reproduced_sha256 == record.sha256_pre_refactor}
        if stem.startswith("ticker_data_flags"):  # the foundation appended two columns; every other byte must be unchanged
            stripped_sha256 = hashlib.sha256(strip_trailing_csv_fields(reproduced_bytes, len(appended_flag_columns))).hexdigest()
            row["sha256_without_appended_columns"] = stripped_sha256
            row["identical_without_appended_columns"] = stripped_sha256 == record.sha256_pre_refactor
        rows.append(row)
    return pd.DataFrame(rows)


**_read_processed_table**

A processed CSV as a frame whose index is the row label (date, ticker, cik or (date, ticker)).


In [22]:
def _read_processed_table(table_path, layout):
    """A processed CSV as a frame whose index is the row label (date, ticker, cik or (date, ticker))."""
    if layout["row"] == "date_and_ticker":
        return pd.read_csv(table_path, index_col=["date", "ticker"], dtype=str)
    return pd.read_csv(table_path, index_col=0, dtype=str)  # text comparison: a change is any change in the written value


**table_cell_changes**

Every cell whose written value differs between two versions of one table, named by ticker, date and column.

Both frames hold text (as written to CSV). A row or column present in only one version is reported with the
missing side as '<absent>'. Returns a frame with columns table, ticker, date, column, before, after.


In [23]:
def table_cell_changes(before, after, table_name, layout):
    """Every cell whose written value differs between two versions of one table, named by ticker, date and column.

    Both frames hold text (as written to CSV). A row or column present in only one version is reported with the
    missing side as '<absent>'. Returns a frame with columns table, ticker, date, column, before, after.
    """
    all_rows = before.index.union(after.index, sort=False)
    all_columns = before.columns.union(after.columns, sort=False)
    before_aligned = before.fillna("").reindex(index=all_rows, columns=all_columns).fillna("<absent>")  # an empty cell is blank; only alignment leaves gaps
    after_aligned = after.fillna("").reindex(index=all_rows, columns=all_columns).fillna("<absent>")
    changed_cells = (before_aligned != after_aligned).stack()
    changed_cells = changed_cells[changed_cells]
    rows = []
    for row_label, column_label in changed_cells.index:
        ticker, date = "", ""
        if layout["row"] == "date":
            date = row_label
        elif layout["row"] == "date_and_ticker":
            date, ticker = row_label
        else:
            ticker = row_label
        if layout["column"] in ("ticker", "cik"):
            ticker = column_label
        rows.append({"table": table_name, "ticker": str(ticker), "date": str(date)[:10], "column": "" if layout["column"] in ("ticker", "cik") else column_label,
                     "before": before_aligned.at[row_label, column_label], "after": after_aligned.at[row_label, column_label]})
    return pd.DataFrame(rows, columns=change_columns)


**processed_table_changes**

table_cell_changes for every table notebook 02 writes, before_folder against after_folder.


In [24]:
def processed_table_changes(before_folder, after_folder):
    """table_cell_changes for every table notebook 02 writes, before_folder against after_folder."""
    all_changes = [table_cell_changes(_read_processed_table(Path(before_folder) / f"{stem}.csv", layout),
                                      _read_processed_table(Path(after_folder) / f"{stem}.csv", layout), stem, layout)
                   for stem, layout in processed_table_layouts.items()]
    return pd.concat(all_changes, ignore_index=True)


**loader_return_changes**

Daily simple returns that differ between corrections off and on (loader panel), as change rows.


In [25]:
def loader_return_changes(through=settings.research_price_cutoff):
    """Daily simple returns that differ between corrections off and on (loader panel), as change rows."""
    returns_off = loader.load_daily_returns(through=through, apply_corrections=False)
    returns_on = loader.load_daily_returns(through=through, apply_corrections=True)
    differs = ~((returns_off == returns_on) | (returns_off.isna() & returns_on.isna()))  # nan on both sides is no change
    changed_cells = differs.stack()
    changed_cells = changed_cells[changed_cells]
    return pd.DataFrame([{"table": "loader:daily_simple_returns", "ticker": ticker, "date": date.strftime("%Y-%m-%d"), "column": "return",
                          "before": repr(float(returns_off.at[date, ticker])), "after": repr(float(returns_on.at[date, ticker]))}
                         for date, ticker in changed_cells.index], columns=change_columns)


**corrections_off_comparison**

Rebuild the processed tables with corrections off in a temporary folder and compare.

Returns (reproduction, correction_effects): the reproduction check against the pre-refactor hashes, and every value
that differs between the corrections-off rebuild and the corrections-on tables in processed_folder, plus the loader's
daily returns. Licensed market cap tables are compared but their values are never copied into the effects table:
a changed licensed cell would appear with before and after withheld.


In [26]:
def corrections_off_comparison(pre_refactor_hashes, processed_folder=None):
    """Rebuild the processed tables with corrections off in a temporary folder and compare.

    Returns (reproduction, correction_effects): the reproduction check against the pre-refactor hashes, and every value
    that differs between the corrections-off rebuild and the corrections-on tables in processed_folder, plus the loader's
    daily returns. Licensed market cap tables are compared but their values are never copied into the effects table:
    a changed licensed cell would appear with before and after withheld.
    """
    processed_folder = settings.processed_data_folder if processed_folder is None else Path(processed_folder)
    with tempfile.TemporaryDirectory() as scratch_folder:  # the rebuild exists only for the comparison
        build_corrections_off_tables(scratch_folder)
        reproduction = reproduction_check(pre_refactor_hashes, scratch_folder)  # files notebook 02 does not write are read from data/processed
        table_changes = processed_table_changes(scratch_folder, processed_folder)
    correction_effects = classify_changes(pd.concat([table_changes, loader_return_changes()], ignore_index=True))
    licensed_rows = correction_effects["table"].isin(licensed_table_stems)
    correction_effects.loc[licensed_rows, ["before", "after", "absolute_difference"]] = np.nan  # licensed values never leave results/licensed
    return reproduction, correction_effects


**classify_changes**

Add absolute_difference and kind to change rows: 'floating_point_rounding' when both values are numbers within
rounding_tolerance of each other (rescaling two prices by one factor can move their ratio in the last binary digit),
otherwise 'correction'.


In [27]:
def classify_changes(changes, rounding_tolerance=rounding_noise_tolerance):
    """Add absolute_difference and kind to change rows: 'floating_point_rounding' when both values are numbers within
    rounding_tolerance of each other (rescaling two prices by one factor can move their ratio in the last binary digit),
    otherwise 'correction'.
    """
    before_numbers = pd.to_numeric(changes["before"], errors="coerce")
    after_numbers = pd.to_numeric(changes["after"], errors="coerce")
    absolute_difference = (after_numbers - before_numbers).abs()
    is_rounding = absolute_difference.notna() & (absolute_difference <= rounding_tolerance)
    return changes.assign(absolute_difference=absolute_difference, kind=np.where(is_rounding, "floating_point_rounding", "correction"))


# ---------- correction sensitivity ----------


**annualised_mean_and_volatility**

(mean x 252, sample standard deviation x sqrt(252)) of a daily return series, missing days ignored.


In [28]:
def annualised_mean_and_volatility(daily_returns):
    """(mean x 252, sample standard deviation x sqrt(252)) of a daily return series, missing days ignored."""
    return (float(daily_returns.mean() * settings.annualisation_sessions),
            float(daily_returns.std(ddof=1) * np.sqrt(settings.annualisation_sessions)))


**twelve_minus_one_momentum**

P(t-21) / P(t-252) - 1 at session position t of one security's adjusted close, and the valid daily returns between.


In [29]:
def twelve_minus_one_momentum(adjusted_close, formation_position):
    """P(t-21) / P(t-252) - 1 at session position t of one security's adjusted close, and the valid daily returns between.

    Returns (momentum, valid_returns_between_endpoints); momentum is nan when either endpoint is missing. This is the
    plain formula for the correction sensitivity only; the study's momentum, with its eligibility rules, is Agent B's.
    """
    recent_position = formation_position - settings.momentum_recent_lag_sessions
    start_position = formation_position - settings.momentum_start_lag_sessions
    if start_position < 0:
        return np.nan, 0
    recent_price = adjusted_close.iloc[recent_position]
    start_price = adjusted_close.iloc[start_position]
    returns_between = adjusted_close.iloc[start_position:recent_position + 1].pct_change(fill_method=None).iloc[1:]
    momentum = recent_price / start_price - 1 if pd.notna(recent_price) and pd.notna(start_price) else np.nan
    return float(momentum), int(returns_between.notna().sum())


**snapshot_portfolio_sensitivity**

Snapshot portfolio return series, corrections on versus off: annualised mean and volatility and each changed day.


In [30]:
def snapshot_portfolio_sensitivity(through=settings.research_price_cutoff):
    """Snapshot portfolio return series, corrections on versus off: annualised mean and volatility and each changed day."""
    snapshot_weights = loader.load_snapshot_weights()
    portfolio_off = build.snapshot_portfolio_daily_returns(loader.load_daily_returns(through=through, apply_corrections=False), snapshot_weights)
    portfolio_on = build.snapshot_portfolio_daily_returns(loader.load_daily_returns(through=through, apply_corrections=True), snapshot_weights)
    returns_off = portfolio_off["snapshot_portfolio_return"]
    returns_on = portfolio_on["snapshot_portfolio_return"]
    first_date, last_date = returns_off.index.min().strftime("%Y-%m-%d"), returns_off.index.max().strftime("%Y-%m-%d")
    mean_off, volatility_off = annualised_mean_and_volatility(returns_off)
    mean_on, volatility_on = annualised_mean_and_volatility(returns_on)
    rows = [
        {"subject": "snapshot_portfolio", "measure": "annualised_mean", "date": "", "window_first_session": first_date, "window_last_session": last_date,
         "corrections_off": mean_off, "corrections_on": mean_on},
        {"subject": "snapshot_portfolio", "measure": "annualised_volatility", "date": "", "window_first_session": first_date, "window_last_session": last_date,
         "corrections_off": volatility_off, "corrections_on": volatility_on},
    ]
    for changed_date in returns_off.index[returns_off != returns_on]:  # the affected dates
        rows.append({"subject": "snapshot_portfolio", "measure": "daily_return", "date": changed_date.strftime("%Y-%m-%d"),
                     "window_first_session": "", "window_last_session": "",
                     "corrections_off": float(returns_off[changed_date]), "corrections_on": float(returns_on[changed_date])})
    return pd.DataFrame(rows)


**back_adjustment_sensitivity**

For each back-adjusted ledger row, the security's annualised volatility over the window_sessions window ending on
each formation date, and its 12-1 momentum, corrections off and on, at every formation whose volatility window or
momentum window [t-252, t] includes the event date. Volatility is of daily simple total returns (a price-only
measure, so the window ends on the formation date itself).


In [31]:
def back_adjustment_sensitivity(ledger=None, window_sessions=settings.main_window_sessions):
    """For each back-adjusted ledger row, the security's annualised volatility over the window_sessions window ending on
    each formation date, and its 12-1 momentum, corrections off and on, at every formation whose volatility window or
    momentum window [t-252, t] includes the event date. Volatility is of daily simple total returns (a price-only
    measure, so the window ends on the formation date itself).
    """
    ledger = corrections.read_extreme_event_ledger() if ledger is None else ledger
    back_adjusted_rows = ledger[ledger["treatment"] == "back_adjust_before_date"]
    panels_off = loader.load_price_panels(apply_corrections=False)
    panels_on = loader.load_price_panels(apply_corrections=True)
    returns_off = loader.daily_simple_returns_from_prices(panels_off.adjusted_close, panels_off.quarantined_returns)
    returns_on = loader.daily_simple_returns_from_prices(panels_on.adjusted_close, panels_on.quarantined_returns)
    sessions = panels_on.adjusted_close.index
    rows = []
    for ledger_row in back_adjusted_rows.itertuples():
        ticker, event_date = ledger_row.ticker, pd.Timestamp(ledger_row.event_date)
        for formation_date in calendar.formation_dates(sessions):
            formation_position = sessions.get_loc(formation_date)
            volatility_window = calendar.estimation_window(sessions, formation_date, window_sessions)
            momentum_window = sessions[max(0, formation_position - settings.momentum_start_lag_sessions): formation_position + 1]
            in_volatility_window = volatility_window.min() <= event_date <= volatility_window.max()
            in_momentum_window = momentum_window.min() <= event_date <= momentum_window.max()
            if not (in_volatility_window or in_momentum_window):
                continue
            _, volatility_off = annualised_mean_and_volatility(returns_off[ticker].reindex(volatility_window))
            _, volatility_on = annualised_mean_and_volatility(returns_on[ticker].reindex(volatility_window))
            momentum_off, valid_returns_off = twelve_minus_one_momentum(panels_off.adjusted_close[ticker], formation_position)
            momentum_on, valid_returns_on = twelve_minus_one_momentum(panels_on.adjusted_close[ticker], formation_position)
            common = {"subject": ticker, "date": formation_date.strftime("%Y-%m-%d"), "event_date": event_date.strftime("%Y-%m-%d")}
            rows.append({**common, "measure": f"annualised_volatility_{window_sessions}", "window_first_session": volatility_window.min().strftime("%Y-%m-%d"),
                         "window_last_session": volatility_window.max().strftime("%Y-%m-%d"), "event_in_window": bool(in_volatility_window),
                         "corrections_off": volatility_off, "corrections_on": volatility_on})
            rows.append({**common, "measure": "momentum_12_1", "window_first_session": momentum_window.min().strftime("%Y-%m-%d"),
                         "window_last_session": momentum_window.max().strftime("%Y-%m-%d"), "event_in_window": bool(in_momentum_window),
                         "corrections_off": momentum_off, "corrections_on": momentum_on,
                         "valid_returns_between_endpoints": valid_returns_on})
    return pd.DataFrame(rows)


**correction_sensitivity**

snapshot_portfolio_sensitivity and back_adjustment_sensitivity in one long table, with the differences.


In [32]:
def correction_sensitivity():
    """snapshot_portfolio_sensitivity and back_adjustment_sensitivity in one long table, with the differences."""
    sensitivity = pd.concat([snapshot_portfolio_sensitivity(), back_adjustment_sensitivity()], ignore_index=True)
    sensitivity["difference"] = sensitivity["corrections_on"] - sensitivity["corrections_off"]
    sensitivity["kind"] = np.select([sensitivity["difference"] == 0, sensitivity["difference"].abs() <= rounding_noise_tolerance],
                                    ["unchanged", "floating_point_rounding"], default="correction")
    leading_columns = ["subject", "measure", "date", "event_date", "event_in_window", "window_first_session", "window_last_session",
                       "corrections_off", "corrections_on", "difference", "kind"]
    return sensitivity.reindex(columns=leading_columns + [column for column in sensitivity.columns if column not in leading_columns])


# ---------- panels, cutoffs, calendar and eligibility ----------


**_shape_row**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [33]:
def _shape_row(table_name, frame):
    frame = frame.to_frame() if isinstance(frame, pd.Series) else frame
    return {"table": table_name, "rows": frame.shape[0], "columns": frame.shape[1],
            "first_date": frame.index.min().strftime("%Y-%m-%d"), "last_date": frame.index.max().strftime("%Y-%m-%d"),
            "non_missing_values": int(frame.notna().to_numpy().sum())}


**panel_shapes_and_date_ranges**

Shape, first and last date and non-missing count of every panel the loader serves at the research cutoff.


In [34]:
def panel_shapes_and_date_ranges(through=settings.research_price_cutoff):
    """Shape, first and last date and non-missing count of every panel the loader serves at the research cutoff."""
    price_panels = loader.load_price_panels(through=through)
    return pd.DataFrame([
        _shape_row("adjusted_close (corrected)", price_panels.adjusted_close),
        _shape_row("split_adjusted_close", price_panels.split_adjusted_close),
        _shape_row("volume", price_panels.volume),
        _shape_row("daily_simple_returns", loader.load_daily_returns(through=through)),
        _shape_row("excess_returns", loader.load_excess_returns(through=through)),
        _shape_row("factors (ff5 + mom + rf)", loader.load_factors(through=through)),
        _shape_row("spy_returns", loader.load_spy_returns(through=through)),
        _shape_row("smh_returns", loader.load_smh_returns(through=through)),
        _shape_row("market_cap_same_security (shape only)", loader.load_market_cap_same_security(through=through)),
        _shape_row("issuer_market_cap (shape only)", loader.load_issuer_market_cap(through=through)),
    ])


**study_cutoffs**

The study's dates: the frozen vintage, the research price cutoff, the last loaded session and the factor cutoff from the data.


In [35]:
def study_cutoffs():
    """The study's dates: the frozen vintage, the research price cutoff, the last loaded session and the factor cutoff from the data."""
    sessions = calendar.trading_sessions()
    return pd.DataFrame([
        {"item": "price_vintage_last_session", "value": settings.price_vintage_last_session, "source": "settings (the frozen pull, named explicitly)"},
        {"item": "research_price_cutoff", "value": settings.research_price_cutoff, "source": "settings"},
        {"item": "last_session_served_by_loader", "value": sessions.max().strftime("%Y-%m-%d"), "source": "loader (sessions on or before the research cutoff)"},
        {"item": "first_session_served_by_loader", "value": sessions.min().strftime("%Y-%m-%d"), "source": "loader"},
        {"item": "factor_cutoff", "value": loader.factor_data_last_date().strftime("%Y-%m-%d"), "source": "read from the factor file (factor_data_last_date)"},
        {"item": "first_formation_date", "value": settings.first_formation_date, "source": "settings"},
    ])


**study_formation_calendar**

calendar.formation_calendar on the loader's sessions with the factor cutoff read from the data.


In [36]:
def study_formation_calendar():
    """calendar.formation_calendar on the loader's sessions with the factor cutoff read from the data."""
    return calendar.formation_calendar(calendar.trading_sessions(), loader.factor_data_last_date())


**eligibility_counts_by_formation**

Per formation, how many securities are eligible for a window_sessions window ending on the factor measurement
cutoff with at least minimum_observations complete joint observations of the excess return and mkt_rf, with the
count for each exclusion reason, the quarantined days the windows hold, and the eligible representative securities.


In [37]:
def eligibility_counts_by_formation(window_sessions, minimum_observations):
    """Per formation, how many securities are eligible for a window_sessions window ending on the factor measurement
    cutoff with at least minimum_observations complete joint observations of the excess return and mkt_rf, with the
    count for each exclusion reason, the quarantined days the windows hold, and the eligible representative securities.
    """
    price_panels = loader.load_price_panels()
    excess_returns = loader.excess_returns(loader.daily_simple_returns_from_prices(price_panels.adjusted_close, price_panels.quarantined_returns),
                                           loader.load_factors()["rf"])
    market_factor = loader.load_factors()["mkt_rf"]
    factor_cutoff = loader.factor_data_last_date()
    is_representative = loader.load_identifiers()["is_representative_security"]
    sessions = price_panels.adjusted_close.index
    rows = []
    for formation_date in calendar.formation_dates(sessions):
        cutoff = calendar.measurement_cutoff(formation_date, factor_cutoff)
        window = calendar.estimation_window(sessions, cutoff, window_sessions)
        window_table = eligibility.panel_window_eligibility(excess_returns, window, minimum_observations, market_factor,
                                                           price_panels.before_same_security_start, price_panels.quarantined_returns)
        reason_counts = window_table["reason"].value_counts()
        rows.append({"formation_date": formation_date.strftime("%Y-%m-%d"), "measurement_cutoff": cutoff.strftime("%Y-%m-%d"),
                     "window_sessions": window_sessions, "minimum_observations": minimum_observations,
                     "window_first_session": window.min().strftime("%Y-%m-%d"), "window_last_session": window.max().strftime("%Y-%m-%d"),
                     "securities": len(window_table), "eligible": int(window_table["eligible"].sum()),
                     "eligible_representative_securities": int((window_table["eligible"] & is_representative.reindex(window_table.index)).sum()),
                     **{f"reason_{reason}": int(reason_counts.get(reason, 0)) for reason in eligibility.eligibility_reasons if reason != eligibility.reason_eligible},
                     "quarantined_returns_in_windows": int(window_table["quarantined_in_window"].sum())})
    return pd.DataFrame(rows)


# ---------- ledger, market cap, manifests and ignore rules ----------


**event_ledger_with_returns**

The ledger joined to the loader's return on each event date with corrections off and on, and a check that every
return beyond the threshold (corrections off, through the research cutoff) has a ledger row.

ledger_return_matches_loader compares the ledger's corrected_return with the corrections-on loader return to 1e-10.


In [38]:
def event_ledger_with_returns(threshold=settings.extreme_return_threshold):
    """The ledger joined to the loader's return on each event date with corrections off and on, and a check that every
    return beyond the threshold (corrections off, through the research cutoff) has a ledger row.

    ledger_return_matches_loader compares the ledger's corrected_return with the corrections-on loader return to 1e-10.
    """
    ledger = corrections.read_extreme_event_ledger()
    returns_off = loader.load_daily_returns(apply_corrections=False)
    returns_on = loader.load_daily_returns(apply_corrections=True)
    extreme_returns = returns_off.stack()
    extreme_returns = extreme_returns[extreme_returns.abs() > threshold]
    ledger_keys = set(zip(ledger["ticker"], pd.to_datetime(ledger["event_date"])))
    table = ledger.copy()
    table["return_corrections_off"] = [float(returns_off.at[pd.Timestamp(date), ticker]) for ticker, date in zip(ledger["ticker"], ledger["event_date"])]
    table["return_corrections_on"] = [float(returns_on.at[pd.Timestamp(date), ticker]) for ticker, date in zip(ledger["ticker"], ledger["event_date"])]
    table["ledger_return_matches_loader"] = (table["corrected_return"] - table["return_corrections_on"]).abs() <= corrections.corrected_return_tolerance
    table["beyond_threshold_corrections_off"] = table["return_corrections_off"].abs() > threshold
    table["beyond_threshold_corrections_on"] = table["return_corrections_on"].abs() > threshold
    unledgered = [(ticker, date) for date, ticker in extreme_returns.index if (ticker, date) not in ledger_keys]
    if unledgered:
        raise ValueError(f"returns beyond +-{threshold:.0%} with no ledger row: {unledgered}")
    return table


**nonpositive_market_cap_handling**

Every (ticker, date) whose raw FMP market cap is zero or negative: how each layer treats it, and the return that day.

Licensed: the table holds no cap values, only how they are treated, but it is still saved under results/licensed.


In [39]:
def nonpositive_market_cap_handling(through=settings.price_vintage_last_session):
    """Every (ticker, date) whose raw FMP market cap is zero or negative: how each layer treats it, and the return that day.

    Licensed: the table holds no cap values, only how they are treated, but it is still saved under results/licensed.
    """
    market_cap_raw = loader._market_cap_raw()
    market_cap_raw = market_cap_raw[market_cap_raw.index <= pd.Timestamp(through)]
    nonpositive_cells = (market_cap_raw <= 0).stack()
    nonpositive_cells = nonpositive_cells[nonpositive_cells]
    descriptive_market_cap = loader.load_market_cap_same_security(through=through, nonpositive_as_missing=False)
    study_market_cap = loader.load_market_cap_same_security(through=through, nonpositive_as_missing=True)
    daily_simple_returns = loader.load_daily_returns(through=through)
    flags_path = settings.processed_data_folder / f"ticker_data_flags_through_{settings.price_vintage_last_session}.csv"
    flags = pd.read_csv(flags_path, index_col="ticker")
    rows = []
    for date, ticker in nonpositive_cells.index:
        rows.append({"ticker": ticker, "date": date.strftime("%Y-%m-%d"),
                     "raw_value_nonpositive": True,
                     "descriptive_table_keeps_nonpositive_value": bool(descriptive_market_cap.at[date, ticker] <= 0),
                     "study_loader_value_missing": bool(pd.isna(study_market_cap.at[date, ticker])),
                     "daily_return_kept": bool(pd.notna(daily_simple_returns.at[date, ticker])),
                     "daily_return": float(daily_simple_returns.at[date, ticker]),
                     "flags_table_market_cap_nonpositive_days": int(flags.at[ticker, "market_cap_nonpositive_days"])})
    return pd.DataFrame(rows)


**manifest_completeness**

For every CSV under the folders: whether its manifest exists, the table's sha256 matches, every input exists and
matches its recorded sha256, and the recorded source code still matches (required for acceptance).

skip_tables names files to leave out, such as the table this check is about to be saved as.


In [40]:
def manifest_completeness(folders=None, skip_tables=()):
    """For every CSV under the folders: whether its manifest exists, the table's sha256 matches, every input exists and
    matches its recorded sha256, and the recorded source code still matches (required for acceptance).

    skip_tables names files to leave out, such as the table this check is about to be saved as.
    """
    folders = [settings.processed_data_folder, settings.corrections_folder, settings.results_folder] if folders is None else folders
    skipped_paths = {Path(table_path).resolve() for table_path in skip_tables}
    rows = []
    for folder in folders:
        for table_path in sorted(Path(folder).rglob("*.csv")):
            if table_path.resolve() in skipped_paths:
                continue
            manifest_path = manifests.manifest_path_for(table_path)
            row = {"table": manifests.repository_relative_path(table_path), "manifest_exists": manifest_path.exists(),
                   "table_sha256_matches": False, "inputs": 0, "inputs_all_exist": False, "inputs_sha256_all_match": False,
                   "mismatched_or_missing_inputs": "", "source_code_files": 0, "source_code_sha256_all_match": True}
            if manifest_path.exists():
                manifest = json.loads(manifest_path.read_text())
                input_entries = manifest.get("inputs", [])
                problems = []
                for entry in input_entries:
                    input_path = settings.repository_root_folder / entry["file"] if not Path(entry["file"]).is_absolute() else Path(entry["file"])
                    if not input_path.exists():
                        problems.append(f"{entry['file']} (missing)")
                    elif manifests.sha256_of_file(input_path) != entry["sha256"]:
                        problems.append(f"{entry['file']} (changed)")
                source_entries = manifest.get("source_code", [])
                source_matches = all((settings.repository_root_folder / entry["file"]).exists()
                                     and manifests.source_entry_matches(entry)
                                     for entry in source_entries)
                row.update({"table_sha256_matches": manifest.get("sha256") == manifests.sha256_of_file(table_path),
                            "inputs": len(input_entries), "inputs_all_exist": not any(problem.endswith("(missing)") for problem in problems),
                            "inputs_sha256_all_match": not problems, "mismatched_or_missing_inputs": "; ".join(problems),
                            "source_code_files": len(source_entries), "source_code_sha256_all_match": source_matches})
            row["complete"] = (row["manifest_exists"] and row["table_sha256_matches"]
                               and row["inputs_sha256_all_match"] and row["source_code_sha256_all_match"])
            rows.append(row)
    return pd.DataFrame(rows)


**git_ignore_listing**

git check-ignore for each path: whether it is ignored, by which rule, and whether that is what was expected.

Uses --no-index so a path is judged by the rules alone, whether or not it is tracked.


In [41]:
def git_ignore_listing(paths_expected_ignored, paths_expected_tracked):
    """git check-ignore for each path: whether it is ignored, by which rule, and whether that is what was expected.

    Uses --no-index so a path is judged by the rules alone, whether or not it is tracked.
    """
    all_paths = list(paths_expected_ignored) + list(paths_expected_tracked)
    completed = subprocess.run(["git", "check-ignore", "--no-index", "--verbose", "--non-matching", *all_paths],
                               cwd=settings.repository_root_folder, capture_output=True, text=True)
    rule_by_path = {}
    for line in completed.stdout.splitlines():  # '<source>:<line>:<pattern>\t<path>', or '::\t<path>' when nothing matches
        rule_text, path_text = line.split("\t", 1)
        rule_by_path[path_text] = rule_text
    rows = []
    for path_text in all_paths:
        rule_text = rule_by_path.get(path_text, "::")
        ignored = rule_text != "::" and not rule_text.split(":", 2)[2].startswith("!")
        expected_ignored = path_text in paths_expected_ignored
        rows.append({"path": path_text, "expected_ignored": expected_ignored, "ignored": ignored,
                     "matched_rule": "" if rule_text == "::" else rule_text, "as_expected": ignored == expected_ignored})
    return pd.DataFrame(rows)


## Calculation source: event sensitivity

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [42]:
"""Bound the DHR distribution decision in the first affected formation estimates.

The primary close-valued correction stays untouched. Alternative and original vendor
returns are diagnostic scenarios, applied only in memory to DHR's one event return.
"""

import sys

import numpy as np
import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.factor_exposures import estimate_factor_models, standard_factor_names
from quiet_index_500.foundation import calendar, loader
from quiet_index_500.research_io import save_table


**distribution_return**

One-session economic wealth return including the marked distribution.


In [43]:
def distribution_return(previous_parent_close, parent_close, distributed_shares, distributed_price):
    """One-session economic wealth return including the marked distribution."""
    return (parent_close + distributed_shares * distributed_price) / previous_parent_close - 1


**event_sensitivity_table**

Both study windows, only formations whose window actually contains the DHR event.


In [44]:
def event_sensitivity_table():
    """Both study windows, only formations whose window actually contains the DHR event."""
    event_date = pd.Timestamp('2016-07-05')
    corrected = loader.load_daily_returns()
    uncorrected = loader.load_daily_returns(apply_corrections=False)
    factors = loader.load_factors()
    common_factors = factors[list(standard_factor_names)].join(loader.load_smh_excess_returns().rename('smh_excess'))
    scenarios = {
        'primary_fortive_first_close': corrected.at[event_date, 'DHR'],
        'original_vendor_double_adjustment': uncorrected.at[event_date, 'DHR'],
        'form_8937_average_valuation': distribution_return(102.32, 80.40, .5, 49.11),
    }
    rows = []
    for formation in calendar.formation_dates(corrected.index):
        cutoff = calendar.measurement_cutoff(formation, loader.factor_data_last_date())
        for length, minimum in [(settings.main_window_sessions, settings.main_window_minimum_observations),
                                (settings.sensitivity_window_sessions, settings.sensitivity_window_minimum_observations)]:
            window = calendar.estimation_window(corrected.index, cutoff, length)
            if event_date not in window:
                continue
            for name, event_return in scenarios.items():
                returns = corrected['DHR'].reindex(window).copy()
                returns.loc[event_date] = event_return
                excess = returns - factors.rf.reindex(window)
                estimate = estimate_factor_models(excess, common_factors.reindex(window), minimum)
                baseline = estimate['models']['ff5_mom']
                augmented = estimate['models']['ff5_mom_smh']
                rows.append({'ticker': 'DHR', 'event_date': event_date, 'formation_date': formation,
                             'measurement_cutoff': cutoff, 'window_sessions': length, 'scenario': name,
                             'event_return': event_return, 'observations': baseline['nobs'],
                             'total_volatility': returns.std(ddof=1)*np.sqrt(settings.annualisation_sessions),
                             'residual_volatility': np.sqrt(baseline['residual_variance']*settings.annualisation_sessions),
                             'm3_mkt_rf': baseline['coefficient'][1],
                             'm4_smh_excess': augmented['coefficient'][-1],
                             'incremental_r_squared': augmented['r_squared']-baseline['r_squared']})
    return pd.DataFrame(rows)


**build_event_sensitivity**

Write the compact scenario table with exact data, ledger and code fingerprints.


In [45]:
def build_event_sensitivity():
    """Write the compact scenario table with exact data, ledger and code fingerprints."""
    from quiet_index_500 import factor_exposures
    table = event_sensitivity_table()
    return save_table(table, 'review', 'event_sensitivity.csv', source_modules=[settings.repository_root_folder / 'src/quiet_index_500/event_sensitivity.py', factor_exposures],
                      inputs=[settings.repository_root_folder/'data/corrections/extreme_event_evidence.md',
                              settings.repository_root_folder/'docs/reviews/foundation_events.md'],
                      note='DHR 2016-07-05 only. Primary close-valued correction versus original vendor error and Form 8937 average valuation; no raw data changed.')


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [46]:
import pandas as pd

from quiet_index_500 import foundation, settings

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)

processed_data_folder = settings.processed_data_folder
foundation_results_folder = settings.results_folder / "foundation"
extreme_event_ledger_path = extreme_event_ledger_path()
raw_price_files = [foundation.loader.adjusted_close_path(), foundation.loader.split_adjusted_close_path(),
                   foundation.loader.volume_path(), foundation.loader.supplement_path()]  # the frozen vintage, named explicitly
notebook_02_tables = [processed_data_folder / f"{table_stem}.csv" for table_stem in foundation.validation.processed_table_layouts]  # what notebook 02 writes

## 1. Reproduction with corrections off, and what the corrections change

Before the shared loader existed, the builder saved the sha256 of every file in `data/processed` (13 October 2026; copied into `results/foundation/pre_refactor_processed_sha256.csv`). Here the build is rerun with corrections off in a temporary folder and each table is compared with those hashes. The six tables notebook 02 does not write are compared as they are on disk. `ticker_data_flags` cannot be byte-identical because the foundation appended two columns (`market_cap_nonpositive_days`, `quarantined_return_days`), so it is also compared with those two trailing fields stripped from every line.

The same corrections-off rebuild is then compared cell by cell with the corrections-on tables in `data/processed`, together with the loader's daily return panel. A change of at most 1e-12 between two numbers is labelled `floating_point_rounding`: multiplying every DHR price before 5 July 2016 by one factor can move the ratio of two such prices in its last binary digit.


In [47]:
pre_refactor_hashes_path = foundation_results_folder / "pre_refactor_processed_sha256.csv"
pre_refactor_hashes = pd.read_csv(pre_refactor_hashes_path)

reproduction, correction_effects = corrections_off_comparison(pre_refactor_hashes)  # one corrections-off rebuild, used for both comparisons

assert reproduction["identical_without_appended_columns"].all()  # with corrections off, every table reproduces (flags after stripping the two new columns)
save_results_table(
    reproduction, "reproduction_check.csv", input_paths=[pre_refactor_hashes_path] + raw_price_files,
    note="Corrections-off rebuild of every table notebook 02 writes (and the six it does not, as on disk) against the sha256 saved before the "
         "shared loader rewrote data/processed. ticker_data_flags is also compared with its two appended columns stripped.")
reproduction.drop(columns=["sha256_pre_refactor", "sha256_reproduced", "sha256_without_appended_columns"])

,file,reproduced_from,identical,identical_without_appended_columns
0,data/processed/ff5_factors_daily_2015-01-02_20...,unchanged file on disk,True,True
1,data/processed/ff5_mom_factors_daily_2015-01-0...,unchanged file on disk,True,True
2,data/processed/issuer_market_cap_through_2026-...,corrections-off rebuild,True,True
3,data/processed/ivv_equity_holdings_2026-09-30.csv,unchanged file on disk,True,True
4,data/processed/market_cap_same_security_throug...,corrections-off rebuild,True,True
5,data/processed/portfolio_exclusions_through_20...,corrections-off rebuild,True,True
6,data/processed/same_security_rules_2026-09-30.csv,corrections-off rebuild,True,True
7,data/processed/sector_weights_2026-09-30.csv,unchanged file on disk,True,True
8,data/processed/smh_daily_returns_through_2026-...,corrections-off rebuild,True,True
9,data/processed/snapshot_portfolio_daily_return...,corrections-off rebuild,True,True


In [48]:
save_results_table(
    correction_effects, "correction_effects.csv", input_paths=notebook_02_tables + [extreme_event_ledger_path] + raw_price_files,
    note="Every value that differs between the corrections-off rebuild and the corrections-on processed tables in data/processed, plus the "
         "loader's daily return panel (table loader:daily_simple_returns). ticker and date name the cell; column is the field. kind is "
         "floating_point_rounding when both values are numbers within 1e-12 of each other, otherwise correction. Licensed market cap values "
         "would be withheld; none changed.")

display(correction_effects.groupby(["table", "kind"]).size().rename("changed_values").to_frame())
correction_effects[correction_effects["kind"] == "correction"]

changed_values
table                                              kind                                   
loader:daily_simple_returns                        correction                            1
                                                   floating_point_rounding              94
snapshot_portfolio_daily_returns_through_2026-1... correction                            1
                                                   floating_point_rounding              30
ticker_data_flags_through_2026-10-12               correction                            2

,table,ticker,date,column,before,after,absolute_difference,kind
30,snapshot_portfolio_daily_returns_through_2026-...,,2016-07-05,snapshot_portfolio_return,-0.005571392577214077,-0.006869026398124719,0.001298,correction
31,ticker_data_flags_through_2026-10-12,DHR,,large_daily_moves,1,0,1.000000,correction
32,ticker_data_flags_through_2026-10-12,DHR,,any_flag,True,False,NaN,correction
127,loader:daily_simple_returns,DHR,2016-07-05,return,0.6121833138802235,0.023260359655981055,0.588923,correction


## 2. Correction sensitivity

Corrections on against off for the constant-weight snapshot portfolio over the research period (2015-01-05 to 2026-09-30), and for DHR at every formation whose 504-session window or 12-1 momentum window contains 5 July 2016. Volatility is the annualised standard deviation of daily simple total returns over the 504 sessions ending on the formation date; momentum is P(t-21)/P(t-252) - 1 on the cleaned adjusted close, computed here only to show the sensitivity (the study's momentum, with its eligibility rules, belongs to the downside and momentum task).


In [49]:
correction_sensitivity = correction_sensitivity()

save_results_table(
    correction_sensitivity, "correction_sensitivity.csv",
    input_paths=[extreme_event_ledger_path, processed_data_folder / f"universe_weights_{settings.holdings_date}.csv"] + raw_price_files,
    note="Corrections on versus off. snapshot_portfolio: annualised mean (mean x 252) and volatility (sd x sqrt 252) of the constant-weight "
         "snapshot return over 2015-01-05 to 2026-09-30, and every day whose return changed (kind floating_point_rounding when within 1e-12). "
         "DHR: annualised volatility over the 504 sessions ending on each formation date, and 12-1 momentum, at every formation whose volatility "
         "or momentum window contains 2016-07-05; event_in_window says which.")

correction_sensitivity[correction_sensitivity["kind"] != "floating_point_rounding"]

,subject,measure,date,event_date,event_in_window,window_first_session,window_last_session,corrections_off,corrections_on,difference,kind,valid_returns_between_endpoints
0,snapshot_portfolio,annualised_mean,,NaN,NaN,2015-01-05,2026-09-30,0.271783,0.271672,-0.000111,correction,NaN
1,snapshot_portfolio,annualised_volatility,,NaN,NaN,2015-01-05,2026-09-30,0.208058,0.208062,0.000004,correction,NaN
32,snapshot_portfolio,daily_return,2016-07-05,NaN,NaN,,,-0.005571,-0.006869,-0.001298,correction,NaN
33,DHR,annualised_volatility_504,2017-12-29,2016-07-05,True,2015-12-31,2017-12-29,0.454738,0.141375,-0.313363,correction,NaN
34,DHR,momentum_12_1,2017-12-29,2016-07-05,False,2016-12-29,2017-12-29,0.206903,0.206903,0.000000,unchanged,231.0
35,DHR,annualised_volatility_504,2018-03-29,2016-07-05,True,2016-03-31,2018-03-29,0.457630,0.150567,-0.307064,correction,NaN
36,DHR,momentum_12_1,2018-03-29,2016-07-05,False,2017-03-29,2018-03-29,0.147510,0.147510,0.000000,unchanged,231.0
37,DHR,annualised_volatility_504,2018-06-29,2016-07-05,True,2016-06-30,2018-06-29,0.459141,0.154935,-0.304205,correction,NaN
38,DHR,momentum_12_1,2018-06-29,2016-07-05,False,2017-06-29,2018-06-29,0.187148,0.187148,0.000000,unchanged,231.0


## 3. Panels, cutoffs and the formation calendar


In [50]:
panel_shapes = panel_shapes_and_date_ranges()
study_cutoffs = study_cutoffs()

save_results_table(panel_shapes, "panel_shapes.csv", input_paths=raw_price_files + [foundation.loader.factors_path()],
                              note="Shape, date range and non-missing count of every panel the loader serves at the 2026-09-30 research cutoff, "
                                   "corrections on. Market cap panels are listed by shape only.")
save_results_table(study_cutoffs, "study_cutoffs.csv", input_paths=[foundation.loader.factors_path(), foundation.loader.adjusted_close_path()],
                              note="The study's dates: the frozen vintage and research cutoff from settings, the sessions the loader serves, and "
                                   "the factor cutoff read from the factor file.")
display(study_cutoffs)
panel_shapes

,item,value,source
0,price_vintage_last_session,2026-10-12,"settings (the frozen pull, named explicitly)"
1,research_price_cutoff,2026-09-30,settings
2,last_session_served_by_loader,2026-09-30,loader (sessions on or before the research cut...
3,first_session_served_by_loader,2015-01-02,loader
4,factor_cutoff,2026-08-31,read from the factor file (factor_data_last_date)
5,first_formation_date,2017-12-29,settings


,table,rows,columns,first_date,last_date,non_missing_values
0,adjusted_close (corrected),2953,503,2015-01-02,2026-09-30,1422426
1,split_adjusted_close,2953,503,2015-01-02,2026-09-30,1422426
2,volume,2953,503,2015-01-02,2026-09-30,1422426
3,daily_simple_returns,2953,503,2015-01-02,2026-09-30,1421894
4,excess_returns,2953,503,2015-01-02,2026-09-30,1411331
5,factors (ff5 + mom + rf),2932,7,2015-01-02,2026-08-31,20524
6,spy_returns,2953,1,2015-01-02,2026-09-30,2952
7,smh_returns,2953,1,2015-01-02,2026-09-30,2952
8,market_cap_same_security (shape only),2953,503,2015-01-02,2026-09-30,1417784
9,issuer_market_cap (shape only),2953,500,2015-01-02,2026-09-30,1409983


In [51]:
formation_calendar = study_formation_calendar()

save_results_table(formation_calendar, "formation_calendar.csv", index=True,
                              input_paths=[foundation.loader.factors_path(), foundation.loader.adjusted_close_path()],
                              note="One row per formation date (last session of each calendar quarter from 2017-12-29): outcome quarter and "
                                   "evaluation block, factor measurement cutoff min(formation, factor cutoff), and whether the outcome quarter "
                                   "is complete for factor-dependent and price-only results.")
formation_calendar

,outcome_quarter,outcome_block,factor_measurement_cutoff,price_measurement_cutoff,outcome_complete_for_factor_results,outcome_complete_for_price_results,outcome_first_session,outcome_last_session,outcome_sessions
formation_date,,,,,,,,,
2017-12-29,2018Q1,development,2017-12-29,2017-12-29,True,True,2018-01-02,2018-03-29,61
2018-03-29,2018Q2,development,2018-03-29,2018-03-29,True,True,2018-04-02,2018-06-29,64
2018-06-29,2018Q3,development,2018-06-29,2018-06-29,True,True,2018-07-02,2018-09-28,63
2018-09-28,2018Q4,development,2018-09-28,2018-09-28,True,True,2018-10-01,2018-12-31,63
2018-12-31,2019Q1,development,2018-12-31,2018-12-31,True,True,2019-01-02,2019-03-29,61
2019-03-29,2019Q2,development,2019-03-29,2019-03-29,True,True,2019-04-01,2019-06-28,63
2019-06-28,2019Q3,development,2019-06-28,2019-06-28,True,True,2019-07-01,2019-09-30,64
2019-09-30,2019Q4,development,2019-09-30,2019-09-30,True,True,2019-10-01,2019-12-31,64
2019-12-31,2020Q1,development,2019-12-31,2019-12-31,True,True,2020-01-02,2020-03-31,62


## 4. Securities eligible at each formation

Eligibility for a regression on `mkt_rf`: the window ends on the factor measurement cutoff, a session counts when both the security's excess return and `mkt_rf` are present, and a window may not reach before the security's same-security start. A quarantined return would count as one missing observation (there are none). The main window is 504 sessions with at least 400 observations; the sensitivity window is 252 sessions with at least 200.


In [52]:
eligibility_counts = pd.concat([
    eligibility_counts_by_formation(settings.main_window_sessions, settings.main_window_minimum_observations),
    eligibility_counts_by_formation(settings.sensitivity_window_sessions, settings.sensitivity_window_minimum_observations),
], ignore_index=True)  # main window first, then the sensitivity window

save_results_table(eligibility_counts, "eligibility_counts_by_formation.csv",
                              input_paths=raw_price_files + [foundation.loader.factors_path(), extreme_event_ledger_path],
                              note="Per formation and window: securities eligible for a regression of excess return on mkt_rf (window ending on "
                                   "min(formation, factor cutoff)), eligible representative securities (one per issuer), the count for each "
                                   "exclusion reason and the quarantined returns inside the windows.")
eligibility_counts.pivot(index="formation_date", columns="window_sessions", values="eligible").rename(columns=lambda sessions: f"eligible_{sessions}")

window_sessions,eligible_252,eligible_504
formation_date,,
2017-12-29,466,461
2018-03-29,468,463
2018-06-29,468,464
2018-09-28,468,466
2018-12-31,469,468
2019-03-29,469,468
2019-06-28,471,468
2019-09-30,472,469
2019-12-31,472,469


## 5. The extreme-event ledger

Thirteen returns beyond plus or minus 50% (corrections off, through 2026-09-30), each with a ledger row; the function raises if any such return lacks one. `ledger_return_matches_loader` checks the ledger's corrected or retained return against the loader with corrections on to 1e-10. The evidence for each row is in `data/corrections/extreme_event_evidence.md`.


In [53]:
event_ledger_check = event_ledger_with_returns()

assert event_ledger_check["ledger_return_matches_loader"].all()
save_results_table(event_ledger_check, "event_ledger_check.csv", input_paths=[extreme_event_ledger_path] + raw_price_files,
                              note="The extreme-event ledger with the loader's return on each event date, corrections off and on, and whether "
                                   "the ledger's corrected_return matches the corrections-on loader to 1e-10.")
event_ledger_check.drop(columns=["price_date", "price_value", "quarantine_dates", "evidence_reference"])

,ticker,event_date,classification,treatment,factor,corrected_return,return_corrections_off,return_corrections_on,ledger_return_matches_loader,beyond_threshold_corrections_off,beyond_threshold_corrections_on
0,AMD,2016-04-22,verified_market_move,retain,NaN,0.522901,0.522901,0.522901,True,True,True
1,DHR,2016-07-05,corporate_action_discrepancy,back_adjust_before_date,1.575536,0.023260,0.612183,0.023260,True,True,False
2,PCG,2019-01-14,verified_market_move,retain,NaN,-0.523593,-0.523593,-0.523593,True,True,True
3,PCG,2019-01-24,verified_market_move,retain,NaN,0.745932,0.745932,0.745932,True,True,True
4,APA,2020-03-09,verified_market_move,retain,NaN,-0.538647,-0.538647,-0.538647,True,True,True
5,OXY,2020-03-09,verified_market_move,retain,NaN,-0.520138,-0.520138,-0.520138,True,True,True
6,TRGP,2020-03-09,verified_market_move,retain,NaN,-0.529074,-0.529074,-0.529074,True,True,True
7,HOOD,2021-08-04,verified_market_move,retain,NaN,0.504060,0.504060,0.504060,True,True,True
8,CVNA,2023-06-08,verified_market_move,retain,NaN,0.560206,0.560206,0.560206,True,True,True
9,GL,2024-04-11,verified_market_move,retain,NaN,-0.531402,-0.531402,-0.531402,True,True,True


## 6. Nonpositive market cap

FMP reports a nonpositive market cap for HONA on 2026-06-26. The descriptive processed table keeps the vendor value and the flags table counts it; the study loader sets it missing for size measurement; the return that day is kept. The table holds no cap values, only how each layer treats them, but it derives from licensed data and is saved under `results/licensed/foundation/`.


In [54]:
nonpositive_market_cap = nonpositive_market_cap_handling()

save_results_table(nonpositive_market_cap, "nonpositive_market_cap_handling.csv", derived_from_licensed_data=True,
                              input_paths=[foundation.loader.market_cap_raw_path(), foundation.loader.adjusted_close_path(),
                                           processed_data_folder / f"ticker_data_flags_through_{settings.price_vintage_last_session}.csv"],
                              note="Every ticker and date with a zero or negative raw FMP market cap, and how each layer treats it. Derived "
                                   "from licensed vendor data: the CSV is git-ignored; this manifest is committed.")
nonpositive_market_cap

,ticker,date,raw_value_nonpositive,descriptive_table_keeps_nonpositive_value,study_loader_value_missing,daily_return_kept,daily_return,flags_table_market_cap_nonpositive_days
0,HONA,2026-06-26,True,True,True,True,0.0,1


## 7. The 2026-10-13 leftovers

The later pull in `data/raw` sits outside the study. It is listed (name, size, sha256) and left in place; nothing parses it.


In [55]:
leftover_files = inventory_leftover_files()

save_results_table(leftover_files, "leftover_files_2026-10-13.csv",
                              note="Files in data/raw from the 2026-10-13 pull, outside the study: listed, left in place, never parsed.")
leftover_files

,path,bytes,sha256
0,data/raw/prices_adjusted_close_through_2026-10...,26039314,7e4dc38838a508a18fbcc455c84e66d6e2d0e3a2fe536a...
1,data/raw/prices_close_split_adjusted_through_2...,25478794,d15f6fbc89ff1cd09f4eedcc2043b2ad177fe52d822dd7...
2,data/raw/prices_through_2026-10-13.manifest.json,1443,0fb459b7dc234d59b93f7a55f9b20a634b4931ef67aee7...
3,data/raw/prices_volume_through_2026-10-13.csv,11506798,9083bfef1c6d2edcc7c51fdff0b9069e7b56f4c4ff9868...


## 8. Ignore rules and manifest completeness

Raw Yahoo prices, raw/processed FMP capitalization panels and the private `local/` folder must be ignored by git. Derived research tables under `results/licensed/` follow the repository owner’s recorded inclusion policy; the directory name records provenance. The listing shows the rule that matched each path. The manifest check then covers every CSV under `data/processed`, `data/corrections` and `results`: the manifest exists, the table's sha256 matches, and every recorded input exists with the recorded sha256. Recorded source code must also match for manifest acceptance. The check leaves out its own output table, which it writes last.


In [56]:
paths_expected_ignored = [
    *[f"data/raw/prices_{field}_through_2026-10-12.csv" for field in ("adjusted_close", "close_split_adjusted", "volume", "supplement", "spy", "smh")],
    *[f"data/raw/prices_{field}_through_2026-10-13.csv" for field in ("adjusted_close", "close_split_adjusted", "volume")],  # the leftovers
    "data/raw/market_cap_fmp_through_2026-10-12.csv",
    "data/processed/market_cap_same_security_through_2026-10-12.csv",
    "data/processed/issuer_market_cap_through_2026-10-12.csv",
    "local/",
    "local/fmp_api_key.txt",
]
paths_expected_tracked = [
    "results/licensed/foundation/nonpositive_market_cap_handling.csv",
    "data/raw/prices_through_2026-10-12.manifest.json",
    "data/raw/market_cap_fmp_through_2026-10-12.manifest.json",
    "data/processed/market_cap_same_security_through_2026-10-12.manifest.json",
    "data/processed/snapshot_portfolio_daily_returns_through_2026-10-12.csv",
    "data/corrections/extreme_event_ledger_2026-09-30.csv",
    "results/foundation/correction_effects.csv",
    "results/licensed/foundation/nonpositive_market_cap_handling.manifest.json",
]
git_ignore_check = git_ignore_listing(paths_expected_ignored, paths_expected_tracked)

assert git_ignore_check["as_expected"].all()
save_results_table(git_ignore_check, "git_ignore_check.csv",
                              note="git check-ignore --no-index --verbose for licensed and raw-price files and local/ (expected ignored) and for "
                                   "their manifests and committable tables (expected not ignored), with the rule that matched.")
git_ignore_check

,path,expected_ignored,ignored,matched_rule,as_expected
0,data/raw/prices_adjusted_close_through_2026-10...,True,True,.gitignore:8:data/raw/prices_*.csv,True
1,data/raw/prices_close_split_adjusted_through_2...,True,True,.gitignore:8:data/raw/prices_*.csv,True
2,data/raw/prices_volume_through_2026-10-12.csv,True,True,.gitignore:8:data/raw/prices_*.csv,True
3,data/raw/prices_supplement_through_2026-10-12.csv,True,True,.gitignore:8:data/raw/prices_*.csv,True
4,data/raw/prices_spy_through_2026-10-12.csv,True,True,.gitignore:8:data/raw/prices_*.csv,True
5,data/raw/prices_smh_through_2026-10-12.csv,True,True,.gitignore:8:data/raw/prices_*.csv,True
6,data/raw/prices_adjusted_close_through_2026-10...,True,True,.gitignore:8:data/raw/prices_*.csv,True
7,data/raw/prices_close_split_adjusted_through_2...,True,True,.gitignore:8:data/raw/prices_*.csv,True
8,data/raw/prices_volume_through_2026-10-13.csv,True,True,.gitignore:8:data/raw/prices_*.csv,True
9,data/raw/market_cap_fmp_through_2026-10-12.csv,True,True,.gitignore:10:data/raw/market_cap_*.csv,True


In [57]:
manifest_completeness_path = foundation_results_folder / "manifest_completeness.csv"
# Audit records describe the previous run and are refreshed by the final audit.
audit_tables = list((foundation_results_folder.parent / "audit").glob("*.csv"))
manifest_completeness = manifest_completeness(
    skip_tables=[manifest_completeness_path, *audit_tables])

save_results_table(manifest_completeness, "manifest_completeness.csv",
                              note="For every non-audit CSV under data/processed, data/corrections and results at the time of the check: manifest present, "
                                   "table sha256 matches, every recorded input present with its recorded sha256, and whether the recorded "
                                   "source code still matches. This table is not in its own listing.")
assert manifest_completeness["complete"].all()
manifest_completeness

,table,manifest_exists,table_sha256_matches,inputs,inputs_all_exist,inputs_sha256_all_match,mismatched_or_missing_inputs,source_code_files,source_code_sha256_all_match,complete
0,data/processed/ff5_factors_daily_2015-01-02_20...,True,True,2,True,True,,0,True,True
1,data/processed/ff5_mom_factors_daily_2015-01-0...,True,True,3,True,True,,0,True,True
2,data/processed/issuer_market_cap_through_2026-...,True,True,10,True,True,,11,True,True
3,data/processed/ivv_equity_holdings_2026-09-30.csv,True,True,4,True,True,,0,True,True
4,data/processed/market_cap_same_security_throug...,True,True,10,True,True,,11,True,True
...,...,...,...,...,...,...,...,...,...,...
93,results/publication/forecast_evidence.csv,True,True,2,True,True,,3,True,True
94,results/review/event_sensitivity.csv,True,True,14,True,True,,25,True,True
95,results/size_liquidity/liquidity.csv,True,True,12,True,True,,23,True,True
96,results/size_liquidity/liquidity_coverage.csv,True,True,12,True,True,,23,True,True


## Conclusions

With corrections switched off, the build reproduces all fourteen tables that were in `data/processed` before the shared loader existed. Thirteen match byte for byte, and `ticker_data_flags` matches byte for byte once the two columns the foundation appended are stripped from each line. The loader is therefore a faithful replacement for the cleaning first written in notebook 02, and every difference that follows comes from the ledger, not from the refactor.

With corrections on, the substantive changes are exactly the ones the ledger intends. DHR's return on 5 July 2016 falls from +61.22% to +2.33%, because the vendor counted the Fortive distribution twice and the back adjustment restores the distribution-inclusive return. The constant-weight snapshot portfolio's return that day moves from -0.557% to -0.687%, which is DHR's 0.21% snapshot weight times the 58.9-point correction after renormalisation. In `ticker_data_flags`, DHR's count of moves beyond 50% goes from 1 to 0 and, since that was its only flag, `any_flag` goes from True to False. Beyond these four values, 94 earlier DHR returns and 30 portfolio returns differ by at most 3e-16. That is floating-point rounding from multiplying both prices of each earlier return by the same factor, not a change of substance. The market cap tables, the exclusion log, SPY, SMH and the same-security rules are unchanged.

The correction barely moves the portfolio but matters a great deal for DHR itself. Over 2015-01-05 to 2026-09-30, the portfolio's annualised mean falls from 27.178% to 27.167% (about one basis point) and its annualised volatility is unchanged to four decimals (20.806%). DHR's annualised volatility over the 504 sessions ending on the first three formations (29 December 2017, 29 March 2018 and 29 June 2018) falls from 45.5%, 45.8% and 45.9% to 14.1%, 15.1% and 15.5%. This is because the single spurious +61% day dominated the variance of a 504-day window. Left uncorrected, DHR's total volatility at those three formations would be about three times its corrected level, and every risk estimate built on those windows would inherit the error. No formation's 12-1 momentum window reaches back to 5 July 2016, so DHR's momentum is identical with and without the correction.

The loader serves 2,953 sessions for the 503 securities, from 2 January 2015 to the research cutoff of 30 September 2026. The factor file ends on 31 August 2026, and that cutoff is read from the data. There are 36 formation dates, from 29 December 2017 to 30 September 2026. Twenty have outcome quarters in the development block, eight in validation and six in confirmation. The last two are labelled incomplete: the 30 June 2026 formation's outcome quarter is complete for price-only results but not for factor-dependent ones, and the 30 September 2026 formation's is complete for neither.

At the first formation, 461 securities are eligible for a 504-session regression on `mkt_rf` with at least 400 observations, rising to 498 at the last formation, whose window ends at the factor cutoff. The 252-session window with at least 200 observations gives 466 and 501. The exclusions are names whose history starts inside the window: same-security starts (six at first, falling to one in the main window and none in the sensitivity window), names with no prices yet and names with too few observations. No return is quarantined, so no window loses an observation to the ledger.

All thirteen ledger rows match the loader to 1e-10. Twelve verified market moves remain beyond plus or minus 50% with corrections on. DHR does not, as intended. HONA's nonpositive FMP market cap on 26 June 2026 is kept in the descriptive table, counted in the flags table and set missing by the study loader, and its return that day is kept. The four files of the 2026-10-13 pull are listed and left in place. Raw prices, raw/processed capitalization panels and `local/` remain ignored. Derived research tables under `results/licensed/` and their manifests are included under the recorded owner policy. Every CSV under `data/processed`, `data/corrections` and `results` has a manifest whose table and input fingerprints match.


## Visual evidence

Source figure: 07_event_sensitivity

These figures use saved results; see the figure guide for sources, sample definitions and limitations.


## Findings

There are 36 formations and 34 complete factor-outcome quarters; the final two formations are descriptive. The 13-event ledger retains verified moves and corrects DHR. Nonpositive HONA capitalization is missing for size without deleting its return.

## Analysis and takeaways

The DHR double-adjusted distribution pushed the first affected residual-volatility estimates to roughly 44.5%, versus 11.4–11.9% after correction. Plausible distribution valuations change the corrected estimate by about 0.03 percentage points. This supports that specific correction, while verified extreme market moves remain in the sample.

## Bridge through the study

04–06 branch from this same foundation. Figure 07 in the report shows why the DHR correction matters for the early factor-risk estimates.
